In [1]:
import pandas as pd
import joblib
import numpy as np
import MetaTrader5 as mt5

In [2]:
mt5.initialize()

True

In [5]:
SYMBOL       = 'EURUSD'
TIMEFRAME    = mt5.TIMEFRAME_H4
VOLUME       = 1.0
DEVIATION    = 10
point        = mt5.symbol_info(SYMBOL).point
# model        = joblib.load(open(r"macd_model.sav","rb"))
STOPLOSS     = 50
TAKEPROFIT   = 100

In [6]:
def market_order(symbol, volume, order_type, sl_pips=None, tp_pips=None, **kwargs):
    """
    Place a market order with dynamically calculated stop loss (SL) and take profit (TP).
    
    Args:
        symbol (str): Trading symbol.
        volume (float): Volume of the trade.
        order_type (str): 'buy' or 'sell'.
        sl_pips (float, optional): Stop loss distance in pips. Default is None.
        tp_pips (float, optional): Take profit distance in pips. Default is None.
        **kwargs: Additional parameters for flexibility.
    
    Returns:
        object: Result of the `mt5.order_send` call.
    """
    tick = mt5.symbol_info_tick(symbol)
    if tick is None:
        print(f"Error: Unable to fetch tick info for symbol {symbol}")
        return None

    # Fetch the symbol's pip size
    symbol_info = mt5.symbol_info(symbol)
    if symbol_info is None:
        print(f"Error: Unable to fetch symbol info for {symbol}")
        return None
    pip_size = symbol_info.point

    order_dict = {'buy': 0, 'sell': 1}
    price_dict = {'buy': tick.ask, 'sell': tick.bid}

    # Calculate SL and TP prices
    sl = None
    tp = None
    if sl_pips is not None:
        sl = price_dict[order_type] - sl_pips * pip_size if order_type == 'buy' else price_dict[order_type] + sl_pips * pip_size
    if tp_pips is not None:
        tp = price_dict[order_type] + tp_pips * pip_size if order_type == 'buy' else price_dict[order_type] - tp_pips * pip_size

    # Construct the request
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": volume,
        "type": order_dict[order_type],
        "price": price_dict[order_type],
        "sl": sl,
        "tp": tp,
        "deviation": DEVIATION,
        "magic": 100,
        "comment": "python market order",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }

    # Send the order request
    order_result = mt5.order_send(request)
    print(order_result)

    return order_result


#---------------------------------------------------------------------------------------------------------------------------------
#In[2] FUNCTION - CLOSING AN OPEN ORDER

def close_order(ticket,symbol):
    positions = mt5.positions_get()

    for pos in positions :
        tick = mt5.symbol_info_tick(pos.symbol)
        type_dict = {0: 1, 1: 0}  # 0 represents buy, 1 represents sell - inverting order_type to close the position
        price_dict = {0: tick.ask, 1: tick.bid}
        
        if pos.ticket==ticket and pos.symbol == symbol:
            request = {
                    "action":mt5.TRADE_ACTION_DEAL,
                    "position":pos.ticket,
                    "symbol":pos.symbol,
                    "volume":pos.volume,
                    "type":type_dict[pos.type],
                    "price":price_dict[pos.type],
                    "deviation": DEVIATION,
                    "magic":100,
                    "comment":"python close order",
                    "type_time":mt5.ORDER_TIME_GTC,
                    "type_filling":mt5.ORDER_FILLING_IOC,
            }           
            order_result = mt5.order_send(request)
            print(order_result)
            
            return order_result
        
    return "Ticket doesn't Exist"  

def get_exposure(symbol):
    positions = mt5.positions_get(symbol=symbol)
    if positions:
        pos_df = pd.DataFrame(positions,columns=positions[0]._asdict().keys())
        exposure = pos_df['volume'].sum()
        
        return exposure

In [8]:
market_order(SYMBOL, VOLUME, 'buy', STOPLOSS, TAKEPROFIT)

OrderSendResult(retcode=10009, deal=945498078, order=1159852310, volume=1.0, price=1.17382, bid=1.17382, ask=1.17382, comment='Request executed', request_id=2737143623, retcode_external=0, request=TradeRequest(action=1, magic=100, order=0, symbol='EURUSD', volume=1.0, price=1.17382, stoplimit=0.0, sl=1.1733200000000001, tp=1.17482, deviation=10, type=0, type_filling=1, type_time=0, expiration=0, comment='python market order', position=0, position_by=0))


OrderSendResult(retcode=10009, deal=945498078, order=1159852310, volume=1.0, price=1.17382, bid=1.17382, ask=1.17382, comment='Request executed', request_id=2737143623, retcode_external=0, request=TradeRequest(action=1, magic=100, order=0, symbol='EURUSD', volume=1.0, price=1.17382, stoplimit=0.0, sl=1.1733200000000001, tp=1.17482, deviation=10, type=0, type_filling=1, type_time=0, expiration=0, comment='python market order', position=0, position_by=0))

In [5]:
df = pd.read_csv("October.csv")

In [6]:
def add_next_close_price_on_signal(df):
    """
    Adds a new column to the DataFrame with the close price of the next signal (1 or -1).
    Sets next_close to 0 if the current signal is 0.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame with 'close' and 'Signal' columns.
    
    Returns:
    pd.DataFrame: The DataFrame with the new column 'next_close'.
    """
    # Initialize a new column
    df['next_close'] = None
    
    # Loop through the DataFrame to find the next signal's close price
    for i in range(len(df)):
        if df.at[i, 'Signal'] == 0:
            df.at[i, 'next_close'] = 0  # Set next_close to 0 if Signal is 0
        elif df.at[i, 'Signal'] in [1, -1]:  # Check for buy or sell signal
            # Find the next row with either a 1 or -1 signal
            for j in range(i + 1, len(df)):
                if df.at[j, 'Signal'] in [1, -1]:
                    df.at[i, 'next_close'] = df.at[j, 'close']
                    break  # Exit the inner loop once the next signal is found

    return df

In [7]:
from functions import *
import joblib
model        = joblib.load(open(r"macd_model.sav","rb"))



def preprocess(df):
    
    df = rsi(df)
    df = calculate_macd(df)
    df = add_time_features(df,'time')
    df = generate_macd_signals(df)
    df = add_next_close_price_on_signal(df)

    df.dropna(inplace=True)
    df.reset_index(inplace=True)


    import pandas as pd
    from sklearn.decomposition import PCA
    from sklearn.preprocessing import StandardScaler

    # Sample data
    # df = pd.read_csv('your_dataset.csv')

    df_alt = df[['open','high','low','close','ema_short','ema_long','next_close']]

    # Step 1: Standardize the data
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_alt)

    # Step 2: Apply PCA
    # Set the number of components you want to retain (e.g., 2 or enough to explain 95% variance)
    pca = PCA(n_components=0.95)  # Retain 95% of the variance
    principal_components = pca.fit_transform(scaled_data)

    # Convert the result to a DataFrame for easier handling
    principal_df = pd.DataFrame(data=principal_components, columns=[f"PC{i+1}" for i in range(principal_components.shape[1])])

    # Explained variance

    # Resulting DataFrame with Principal Components
    df['PCA'] = principal_df['PC1']
    
    pred = model.predict(df[['PCA','RSI', 'Signal','MACD','signal_line','MACD_histogram','month','hour','minute','day','year']])
    df['Preds'] = pred
    print(df)
    return df
    

In [9]:
gg = preprocess(df[:100])
gg['Signal'].value_counts()

    index  Unnamed: 0                time     open     high      low    close  \
0       1           1 2022-01-03 01:05:00  1.13782  1.13782  1.13742  1.13763   
1       2           2 2022-01-03 01:10:00  1.13762  1.13763  1.13728  1.13730   
2       3           3 2022-01-03 01:15:00  1.13729  1.13762  1.13713  1.13718   
3       4           4 2022-01-03 01:20:00  1.13718  1.13721  1.13697  1.13705   
4       5           5 2022-01-03 01:25:00  1.13705  1.13736  1.13705  1.13734   
..    ...         ...                 ...      ...      ...      ...      ...   
93     94          94 2022-01-03 08:50:00  1.13460  1.13463  1.13442  1.13442   
94     95          95 2022-01-03 08:55:00  1.13442  1.13444  1.13430  1.13431   
95     96          96 2022-01-03 09:00:00  1.13432  1.13448  1.13406  1.13432   
96     97          97 2022-01-03 09:05:00  1.13432  1.13453  1.13399  1.13451   
97     98          98 2022-01-03 09:10:00  1.13451  1.13462  1.13438  1.13441   

    tick_volume  spread  re

f:\BullsEye - v5\functions.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['RSI'] = rsi
f:\BullsEye - v5\functions.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['ema_short'] = data['close'].ewm(span=short_period, adjust=False).mean()
f:\BullsEye - v5\functions.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-do

Signal
-1    55
 1    43
Name: count, dtype: int64

In [10]:
def has_active_trade(symbol):
    """
    Check if there is an active trade for the given symbol.
    
    Args:
        symbol (str): The trading symbol to check.

    Returns:
        bool: True if there is an active trade for the symbol, False otherwise.
    """
    positions = mt5.positions_get(symbol=symbol)
    return len(positions) > 0 if positions else False


def close_opposite_positions(symbol, direction):
    """
    Close all opposite positions for a given symbol.
    
    Args:
        symbol (str): The trading symbol to check.
        direction (str): The direction of the new order ('buy' or 'sell').

    Returns:
        None
    """
    positions = mt5.positions_get(symbol=symbol)
    if positions is None:
        logger.error(f"Error retrieving positions for {symbol}: {mt5.last_error()}")
        return
    
    opposite_type = 1 if direction == "buy" else 0  # Close 'sell' if direction is 'buy', and vice versa

    for position in positions:
        if position.type == opposite_type:  # Check if the position is of the opposite type
            result = close_order(position.ticket,symbol)
            return result


In [ ]:
import MetaTrader5 as mt5
import logging
import time
import pandas as pd
from functions import *

# Logging setup
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Function to reset log file handler
def reset_log_file():
    global file_handler
    logger.removeHandler(file_handler)
    file_handler = logging.FileHandler('tick_check.txt', mode='w')  # 'w' to truncate the file
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

# File Handler to log to a file
file_handler = logging.FileHandler('tick_check.txt', mode='w')  # Start with a fresh file
file_handler.setLevel(logging.INFO)

# Console Handler to log to the console
console_handler = logging.StreamHandler()  # Prints logs to console
console_handler.setLevel(logging.INFO)

# Define the log format
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

# Add handlers to the logger
logger.addHandler(file_handler)
logger.addHandler(console_handler)

# Global variables to keep track of the previous tick
previous_tick = None

def check_tick(symbol):
    global previous_tick

    # Fetch the latest tick
    tick = mt5.symbol_info_tick(symbol)
    
    if tick is None:
        logger.warning(f"Unable to fetch tick for {symbol}")
        return False
    
    # Log if a new tick has occurred
    if previous_tick is None:
        previous_tick = tick
        logger.info(f"First tick received for {symbol}: Ask={tick.ask}, Bid={tick.bid}")
        return True
    
    # Check if the tick has changed
    if tick.ask != previous_tick.ask or tick.bid != previous_tick.bid:
        logger.info(f"New tick received for {symbol}: Ask={tick.ask}, Bid={tick.bid}")
        previous_tick = tick
        return True
    else:
        logger.debug(f"No change in tick for {symbol}: Ask={tick.ask}, Bid={tick.bid}")
        return False

def main(symbol):
    # Ensure MetaTrader 5 is initialized and connected
    if not mt5.initialize():
        logger.error("MetaTrader5 initialization failed")
        return
    
    try:
        last_reset_time = time.time()  # Track the time of the last reset
        
        while True:
            current_time = time.time()
            
            # Reset the log file every 60 seconds
            if current_time - last_reset_time >= 60:
                reset_log_file()
                last_reset_time = current_time
                logger.info("Log file cleared and reset.")
            
            tick_occurred = check_tick(symbol)
            if tick_occurred:
                logger.info(f"Tick occurred for {symbol}")
                
                # Fetch the last 100 minutes data
                rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_M5, 1, 100)  # 100 data points (5-minute intervals)
                if rates is None or len(rates) < 100:
                    logger.warning("Not enough data to process")
                    continue
                
                df = pd.DataFrame(rates)
                df['time'] = pd.to_datetime(df['time'], unit='s')

                # Preprocess the data to generate features
                df = preprocess(df)
                
                # Make prediction using the preprocessed data
                prediction = df['Preds'].iloc[-1]  # Assuming the prediction is in the 'Preds' column
                signal = df['Signal'].iloc[-1]
                logger.info(f"SIGNAL = {signal}")
                if prediction == 1:
                    
                    logger.info(f"Prediction is Profit: Place trade {signal}")
                    if signal == 1:  # Buy Signal
                        if has_active_trade(SYMBOL):
                            close_opposite_positions(SYMBOL, "buy")
                        if not has_active_trade(SYMBOL):  # Ensure no active trades remain after closing
                            market_order(SYMBOL, VOLUME, "buy", STOPLOSS, TAKEPROFIT)
                        else:
                            logger.info(f"Active trade already exists for {SYMBOL}. Skipping buy order.")

                    else:  # Sell Signal
                        if has_active_trade(SYMBOL):
                            close_opposite_positions(SYMBOL, "sell")
                        if not has_active_trade(SYMBOL):  # Ensure no active trades remain after closing
                            market_order(SYMBOL, VOLUME, "sell", STOPLOSS, TAKEPROFIT)
                        else:
                            logger.info(f"Active trade already exists for {SYMBOL}. Skipping sell order.")
                    # Add trade execution code here (e.g., place a buy order)
                else:
                    logger.info(f"Prediction is Loss: Don't Place Trade {signal}")
                        
                    # Add trade execution code here (e.g., place a sell order)
            
            else:
                logger.debug(f"No tick update for {symbol}")
            
            time.sleep(1)  # Wait for a second before checking again

    except KeyboardInterrupt:
        logger.info("Tick check process interrupted by user.")
    
    finally:
        # Shutdown the connection to MT5
        mt5.shutdown()

if __name__ == "__main__":
    symbol = "EURUSD"  # Replace with the desired symbol (e.g., "GBPUSD")
    print(f"Tracking ticks for: {symbol}")
    main(symbol)


In [ ]:
def check_macd_signal(df):
    """
    Check MACD crossover signal and return:
    - 1 (Buy Signal) if the crossover occurred in the last candle,
    - -1 (Sell Signal) if the crossover occurred in the last candle,
    - 0 (No Signal) if no recent crossover.
    Also prints how many candles ago the signal occurred.
    """
    signal_candles_ago = None
    
    for i in range(len(df) - 2, -1, -1):  # Start checking from the second-last candle
        macd_prev = df['MACD'].iloc[i]
        signal_prev = df['signal_line'].iloc[i]
        macd_curr = df['MACD'].iloc[i + 1]
        signal_curr = df['signal_line'].iloc[i + 1]
        
        # Check for bullish crossover (MACD crossing above the Signal Line)
        if macd_prev < signal_prev and macd_curr > signal_curr:
            signal_candles_ago = len(df) - i - 2  # Calculate how many candles ago the signal was
            # logger.info(f"Bullish MACD crossover occurred {signal_candles_ago} candle(s) ago.")
            return 1 if signal_candles_ago == 1 else 0  # Trade only if the signal was on the previous candle
        
        # Check for bearish crossover (MACD crossing below the Signal Line)
        if macd_prev > signal_prev and macd_curr < signal_curr:
            signal_candles_ago = len(df) - i - 2  # Calculate how many candles ago the signal was
            # logger.info(f"Bearish MACD crossover occurred {signal_candles_ago} candle(s) ago.")
            return -1 if signal_candles_ago == 1 else 0  # Trade only if the signal was on the previous candle
    
    logger.info("No recent MACD crossover signal detected.")
    return 0 # No Signal

In [8]:
rates = mt5.copy_rates_from_pos("EURUSD", mt5.TIMEFRAME_M5, 1, 100)  # 100 data points (5-minute intervals)
if rates is None or len(rates) < 100:
    logger.warning("Not enough data to process")

df = pd.DataFrame(rates)
df['time'] = pd.to_datetime(df['time'], unit='s')

# Preprocess the data to generate features
df = preprocess(df)

    index                time     open     high      low    close  \
0       1 2024-11-19 21:50:00  1.05913  1.05916  1.05890  1.05907   
1       2 2024-11-19 21:55:00  1.05907  1.05951  1.05905  1.05935   
2       3 2024-11-19 22:00:00  1.05934  1.05947  1.05913  1.05937   
3       4 2024-11-19 22:05:00  1.05937  1.05950  1.05936  1.05950   
4       5 2024-11-19 22:10:00  1.05950  1.05960  1.05939  1.05955   
..    ...                 ...      ...      ...      ...      ...   
93     95 2024-11-20 05:40:00  1.05926  1.05926  1.05912  1.05917   
94     96 2024-11-20 05:45:00  1.05917  1.05917  1.05904  1.05904   
95     97 2024-11-20 05:50:00  1.05903  1.05904  1.05884  1.05887   
96     98 2024-11-20 05:55:00  1.05888  1.05898  1.05883  1.05896   
97     99 2024-11-20 06:00:00  1.05897  1.05901  1.05890  1.05892   

    tick_volume  spread  real_volume        RSI  ...  MACD_histogram  minute  \
0           105       4            0   0.000000  ...       -0.000004      50   
1          

In [12]:
df.columns

Index(['index', 'time', 'open', 'high', 'low', 'close', 'tick_volume',
       'spread', 'real_volume', 'RSI', 'ema_short', 'ema_long', 'MACD',
       'signal_line', 'MACD_histogram', 'minute', 'hour', 'day', 'month',
       'year', 'Signal', 'next_close', 'PCA', 'Preds'],
      dtype='object')

In [14]:
gg = check_macd_signal(df)

NameError: name 'logger' is not defined